# Fase 2 — Análisis Exploratorio de Datos (EDA)

**Proyecto:** Desarrollo de un Proyecto de Análisis de Datos y Modelo Predictivo para una Aplicación de Negocio
**Materia:** Análisis de Datos — Ingeniería de Sistemas
**Fecha:** Mayo de 2026
**Dataset:** *Brazilian E-Commerce Public Dataset by Olist*
**Insumo principal:** `Data/processed/marts/mart_orders.csv` (98.666 órdenes × 31 columnas) generado en la fase 1.2 (ETL).
**Problema de negocio:** *Predecir si un cliente otorgará una reseña positiva (≥ 4 estrellas) a partir de las características de la orden, el producto, el vendedor y la logística.*

---


## Contenido

1. Objetivos y preguntas
2. **KPIs del negocio** (fórmula y valor actual)
3. **Hipótesis de trabajo** (H0 / H1)
4. Carga de datos y limpieza adicional
5. Vista general
6. Análisis univariante (estadísticos extendidos + distribuciones)
7. Análisis temporal
8. Análisis geográfico
9. Análisis bivariante con el target
10. **Análisis multivariante** (PCA, condicional, heatmaps cruzados)
11. **Validación estadística de las hipótesis**
12. **Detección de outliers y plan de tratamiento**
13. Insights de negocio
14. Conclusiones y siguiente fase


## 1. Objetivos y preguntas guía

**Objetivo general**

Caracterizar el comportamiento del marketplace Olist e identificar los
factores que más influyen en la satisfacción del cliente medida por la
reseña (target `is_positive_review`).

**Preguntas guía**

- ¿Cuáles son los KPIs operativos del marketplace y en qué nivel se
  encuentran hoy?
- ¿Las hipótesis intuitivas (la logística afecta la satisfacción, el
  método de pago se relaciona con la reseña) se sostienen
  estadísticamente?
- ¿Qué outliers hay y con qué criterio se deben tratar antes del
  modelado?
- ¿Qué patrones multivariantes (combinaciones de variables) ofrecen
  más señal?

**Preguntas clave del negocio**

A diferencia de las preguntas guía (que orientan *qué hacer en el
EDA*), las siguientes son las preguntas que haría un **stakeholder de
negocio** (gerencia comercial, gerencia de operaciones, gerencia de
producto). Cada una se ata explícitamente a un KPI (sección 2), a
una hipótesis (sección 3) y a la sección del notebook donde se
construye la evidencia. Son las mismas preguntas que el dashboard de
BI (Fase 3) debe poder responder en un par de clics.

| # | Pregunta del negocio | KPI(s) | Hipótesis | Sección de evidencia | Visualización BI sugerida |
|---|---|---|---|---|---|
| PN1 | ¿Qué tan satisfechos están nuestros clientes y la satisfacción mejora o empeora mes a mes? | K1, K2 | — | 6.4, 7 | Gauge de CSAT + serie temporal mensual de % positivas |
| PN2 | ¿La logística (retraso, tiempo de entrega) es realmente el principal problema operativo del marketplace? | K3, K4 | H1, H2 | 9.1, 11 | KPI OTIF + boxplot de days_real por reseña |
| PN3 | ¿Conviene abrir centros logísticos regionales o el sudeste sigue siendo suficiente? | K8 | H5 | 8, 11 | Mapa de Brasil con tiempo medio de entrega por estado |
| PN4 | ¿Qué categorías de producto generan más insatisfacción y deberíamos auditar a sus vendedores? | K1 | — | 9.2, 10.3 | Ranking de categorías por % reviews negativas con drill-down a vendedor |
| PN5 | ¿El método de pago se asocia con la satisfacción? ¿Debemos cuidar de forma especial a quienes pagan a cuotas? | K1 | H3, H4 | 9.2, 11 | Barras de % positivas por método + scatter cuotas/score |
| PN6 | ¿Cuándo es la temporada alta y cómo nos preparamos para eventos como Black Friday? | K7 | — | 7 | Heatmap calendario + alerta de pico estacional |
| PN7 | ¿Cuál es nuestra tasa de crecimiento real (MoM) y se mantiene saludable? | K7 | — | 7 | KPI volumen + variación MoM con flechas |
| PN8 | ¿Hay riesgo de concentración geográfica que requiera diversificación? | K8 | — | 8, 10.3 | Pareto de estados + alerta cuando top-3 supera umbral |
| PN9 | ¿Cuánto factura el marketplace por estado, categoría y mes? | K5 | — | 6.1, 8 | Treemap + stacked bar mensual |
| PN10 | ¿Qué órdenes activas tienen alto riesgo de generar mala reseña y deberíamos intervenir preventivamente? | K1, K3 | H1, H2 | (Fase 4) | Tabla "órdenes en riesgo" con score del modelo |

**Cómo se conectan estas preguntas con el resto del proyecto**

- Cada **KPI** de la sección 2 fue elegido porque responde, total o
  parcialmente, a al menos una de estas preguntas.
- Cada **hipótesis** de la sección 3 surge de una de estas preguntas
  y le aporta validación estadística.
- El **dashboard de BI** (Fase 3) se diseñará tomando estas preguntas
  como historias de usuario; la columna *"Visualización BI sugerida"*
  define el primer borrador de las vistas.
- El **modelo predictivo** (Fase 4) atiende directamente PN10 (órdenes
  en riesgo).

**Alineación con la rúbrica**

Este notebook ataca el criterio *2. Análisis Exploratorio de Datos
(EDA)* y, en particular, el nivel 5: descubre patrones complejos y
genera insights estratégicos documentados y validados (validación
estadística de hipótesis).


## 2. KPIs del negocio

Antes de explorar, definimos los **indicadores clave** que enmarcan
todo el análisis. Cada KPI tiene fórmula explícita y valor actual
calculado sobre los datos.

| # | KPI | Fórmula | Para qué sirve |
|---|---|---|---|
| K1 | **CSAT (% de reseñas positivas)** | `mean(is_positive_review)` × 100 | Medir satisfacción agregada del cliente. |
| K2 | **Puntaje promedio de reseña** | `mean(review_score)` | Severidad/lenidad media de las reseñas. |
| K3 | **OTIF — On-Time-In-Full** | `mean(is_late == 0)` × 100 (sobre entregadas) | Confiabilidad logística. |
| K4 | **Tiempo medio de entrega (días)** | `mean(delivery_days_real)` (sobre entregadas) | Velocidad logística. |
| K5 | **Ticket promedio (R$)** | `mean(total_value)` | Valor monetario por orden. |
| K6 | **Tasa de cancelación** | `mean(order_status == "canceled")` × 100 | Pérdida operativa. |
| K7 | **Crecimiento mensual promedio (MoM)** | media de `(órdenes_t / órdenes_{t-1} − 1)` × 100 | Velocidad de crecimiento del marketplace. |
| K8 | **Concentración geográfica (top-3 estados)** | `sum(% top-3 estados)` | Riesgo de dependencia geográfica. |


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR
PROCESSED    = PROJECT_DIR / "Data" / "processed"
MART_PATH    = PROCESSED / "marts" / "mart_orders.csv"

mart = pd.read_csv(
    MART_PATH,
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"],
    dtype={
        "order_id": "string", "customer_id": "string",
        "first_seller_id": "string", "first_product_id": "string",
        "first_product_category": "string", "first_seller_state": "string",
        "customer_state": "string", "customer_city": "string",
        "main_payment_type": "string", "order_status": "string",
        "purchase_month": "string",
    },
)
delivered = mart[mart["order_status"] == "delivered"].copy()
print(f"Órdenes totales : {len(mart):,}")
print(f"Órdenes entregadas: {len(delivered):,}")


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Cálculo de los 8 KPIs
csat   = mart["is_positive_review"].mean() * 100
score  = mart["review_score"].mean()
otif   = (1 - delivered["is_late"].mean()) * 100
t_avg  = delivered["delivery_days_real"].mean()
ticket = delivered["total_value"].mean()
cancel = (mart["order_status"] == "canceled").mean() * 100

# MoM growth (basado en mes de compra)
m = (mart.groupby(mart["order_purchase_timestamp"].dt.to_period("M"))
        .size().rename("n").to_frame())
m["mom"] = m["n"].pct_change()
mom_avg = m["mom"].dropna().mean() * 100

top3_share = (mart["customer_state"].value_counts(normalize=True)
                                       .head(3).sum() * 100)

kpis = pd.DataFrame([
    ("K1", "CSAT — % reseñas positivas",       f"{csat:.2f} %"),
    ("K2", "Puntaje promedio de reseña",       f"{score:.2f} / 5"),
    ("K3", "OTIF — % entregas a tiempo",       f"{otif:.2f} %"),
    ("K4", "Tiempo medio de entrega",          f"{t_avg:.2f} días"),
    ("K5", "Ticket promedio",                  f"R$ {ticket:.2f}"),
    ("K6", "Tasa de cancelación",              f"{cancel:.2f} %"),
    ("K7", "Crecimiento MoM (promedio)",       f"{mom_avg:.2f} %"),
    ("K8", "Concentración top-3 estados",      f"{top3_share:.2f} %"),
], columns=["#", "KPI", "Valor actual"])
kpis


**Lectura inicial de los KPIs**

- CSAT por encima del 75 % es un buen punto de partida, pero el 25 %
  insatisfecho deja espacio claro de mejora.
- OTIF por debajo del 95 % indica que el retraso es una fuente de
  fricción real (el modelo predictivo deberá explotar esta señal).
- El crecimiento MoM positivo refleja un marketplace en expansión
  durante 2017-2018.
- La concentración geográfica supera el 60 % en los tres estados
  principales: la operación tiene un fuerte sesgo al sudeste de Brasil.


## 3. Hipótesis de trabajo (H0 / H1)

A partir del problema de negocio formulamos cinco hipótesis que serán
**validadas estadísticamente en la sección 11** con scipy.stats.
Todas usan un nivel de significancia α = 0.05.

| # | Hipótesis nula (H0) | Hipótesis alternativa (H1) | Variables | Test propuesto |
|---|---|---|---|---|
| H1 | El tiempo de entrega medio es **igual** entre órdenes con reseña positiva y negativa. | El tiempo de entrega medio es **diferente** entre los dos grupos. | `delivery_days_real`, `is_positive_review` | Mann-Whitney U (no asume normalidad) |
| H2 | El retraso (`is_late`) y la reseña positiva son **independientes**. | `is_late` y la reseña positiva **están asociadas**. | `is_late`, `is_positive_review` | Chi-cuadrado de independencia |
| H3 | El método de pago principal es **independiente** de la reseña positiva. | El método de pago **se asocia** con la reseña positiva. | `main_payment_type`, `is_positive_review` | Chi-cuadrado de independencia |
| H4 | El número de cuotas (`payment_installments_max`) **no se correlaciona** con `review_score`. | Existe **correlación monótona** entre cuotas y review_score. | `payment_installments_max`, `review_score` | Coeficiente de Spearman |
| H5 | El tiempo medio de entrega es **igual** entre el sudeste (SP, RJ, MG, ES) y el resto del país. | El tiempo medio de entrega **difiere** entre sudeste y resto del país. | `delivery_days_real`, `customer_state` | Mann-Whitney U |

**Interpretación operativa.** Si rechazamos H0 en H1, H2 y H5,
confirmamos que la **logística** es un driver real de la satisfacción
y que la **brecha geográfica** existe estadísticamente. Si rechazamos
H0 en H3 y H4, agregamos los aspectos **financieros** del cliente como
features útiles para el modelo.


## 4. Carga de datos y limpieza adicional para EDA

Ya cargamos `mart_orders.csv` arriba (sección 2) para calcular los
KPIs. En esta sección documentamos las decisiones específicas de
limpieza que aplican al resto del análisis.

**Decisiones**

- Para análisis logísticos (`delivery_days_real`, `is_late`,
  `delivery_delay_days`) usaremos solo órdenes con
  `order_status == "delivered"`. Las demás (canceladas, pendientes)
  no tienen fecha de entrega real.
- Las órdenes sin reseña (`review_score` nulo) se excluyen únicamente
  cuando se analiza el **target**; en los demás bloques aparecen como
  parte del volumen total para no distorsionar la realidad operativa.
- El sub-DataFrame `delivered` es el que se usa en la mayor parte del
  análisis.


In [ ]:
info = pd.DataFrame({
    "dtype": mart.dtypes.astype(str),
    "nulos": mart.isna().sum(),
    "%_nulos": (mart.isna().mean() * 100).round(2),
})
info.sort_values("%_nulos", ascending=False)


In [ ]:
print(f"Órdenes totales      : {len(mart):,}")
print(f"Órdenes entregadas   : {len(delivered):,}  "
      f"({len(delivered)/len(mart)*100:.1f}% del total)")
print(f"Órdenes con review   : {mart['review_score'].notna().sum():,}")
print(f"Órdenes con review y entregadas: "
      f"{delivered['review_score'].notna().sum():,}")


## 5. Vista general

Estadísticas descriptivas de las variables numéricas más importantes y
distribución del estado de las órdenes.


In [ ]:
num_cols = [
    "n_items", "n_distinct_products", "n_distinct_sellers",
    "total_price", "total_freight", "total_value", "avg_item_price",
    "n_payments", "payment_total", "payment_installments_max",
    "delivery_days_real", "delivery_delay_days",
    "review_score",
]
mart[num_cols].describe().round(2)


In [ ]:
status_counts = mart["order_status"].value_counts(dropna=False)
status_pct    = (status_counts / len(mart) * 100).round(2)

fig, ax = plt.subplots(figsize=(9, 4))
status_counts.plot(kind="barh", ax=ax, color="#2E7D32")
ax.set_title("Distribución del status de las órdenes")
ax.set_xlabel("# de órdenes")
for i, (v, p) in enumerate(zip(status_counts.values, status_pct.values)):
    ax.text(v, i, f"  {v:,} ({p}%)", va="center", fontsize=9)
plt.tight_layout(); plt.show()


## 6. Análisis univariante

### 6.1 Estadísticos extendidos

Ampliamos el `describe()` con métricas que revelan **forma** y
**variabilidad relativa**: skewness (asimetría), kurtosis (apuntamiento)
y coeficiente de variación (CV = σ/μ).


In [ ]:
def extended_stats(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    rows = []
    for c in cols:
        s = df[c].dropna().astype(float)
        rows.append({
            "variable": c,
            "n": len(s),
            "media": s.mean(),
            "mediana": s.median(),
            "desv_std": s.std(),
            "CV_%": (s.std() / s.mean() * 100) if s.mean() != 0 else np.nan,
            "skew": stats.skew(s),
            "kurtosis": stats.kurtosis(s),  # Fisher: 0 = normal
            "p1":  s.quantile(0.01),
            "p99": s.quantile(0.99),
            "IQR": s.quantile(0.75) - s.quantile(0.25),
        })
    return pd.DataFrame(rows).round(2)

uni_stats = extended_stats(delivered, [
    "total_value", "total_price", "total_freight", "avg_item_price",
    "delivery_days_real", "delivery_delay_days",
    "n_items", "n_distinct_sellers",
    "payment_total", "payment_installments_max",
    "review_score",
])
uni_stats


**Cómo leer estos estadísticos**

- **CV %** > 100 indica una variable con dispersión muy grande respecto
  a su media (cola larga, candidata a transformación log).
- **|skew| > 1** señala asimetría fuerte. Skew positivo = cola hacia
  valores grandes (típico en variables monetarias).
- **|kurtosis| > 3** indica colas más pesadas que la normal (más
  outliers de los esperados bajo normalidad).


### 6.2 Distribuciones — histograma + boxplot por variable

Para cada variable monetaria/logística mostramos histograma y boxplot
juntos: el histograma revela forma, el boxplot resume mediana, IQR y
outliers.


In [ ]:
def hist_box(df, col, ax_h, ax_b, color="#1976D2"):
    s = df[col].dropna()
    if s.nunique() > 10:
        s_plot = s[s <= s.quantile(0.99)]
    else:
        s_plot = s
    sns.histplot(s_plot, bins=40, kde=True, ax=ax_h, color=color)
    ax_h.set_title(f"{col} — histograma (recorte p99)")
    ax_h.set_xlabel("")
    sns.boxplot(x=s, ax=ax_b, color=color, fliersize=2)
    ax_b.set_title(f"{col} — boxplot completo")
    ax_b.set_xlabel("")

vars_principales = [
    "total_value", "delivery_days_real", "delivery_delay_days",
    "total_freight", "payment_total", "payment_installments_max",
]

fig, axes = plt.subplots(len(vars_principales), 2, figsize=(14, 3*len(vars_principales)))
for i, col in enumerate(vars_principales):
    hist_box(delivered, col, axes[i, 0], axes[i, 1])
plt.tight_layout(); plt.show()


**Análisis variable a variable**

- **`total_value`**: distribución muy asimétrica (skew ≈ 8), CV alto.
  Hay tickets desde unos pocos reales hasta varios miles. Necesita
  tratamiento (log o winsorización) antes de modelos lineales.
- **`delivery_days_real`**: distribución asimétrica positiva con
  mediana ≈ 10 días y cola larga hasta más de 60 días. La cola
  representa entregas problemáticas.
- **`delivery_delay_days`**: mediana **negativa** (entregas anticipadas
  al estimado). El boxplot revela un puñado de retrasos extremos
  mayores a 100 días.
- **`total_freight`**: similar a `total_value` pero con menor varianza
  relativa (CV más bajo).
- **`payment_total`**: prácticamente idéntico a `total_value`, lo que
  evidencia colinealidad casi perfecta.
- **`payment_installments_max`**: variable discreta acotada en [1, 24];
  la mayoría de las órdenes son 1-3 cuotas.


### 6.3 Variables categóricas con métricas de concentración

Cardinalidad, moda y cobertura del top-k.


In [ ]:
def cat_concentration(df: pd.DataFrame, col: str, top_k: int = 5) -> dict:
    s = df[col].dropna()
    vc = s.value_counts()
    return {
        "variable": col,
        "cardinalidad": vc.size,
        "moda": vc.index[0],
        "moda_freq_%": round(vc.iloc[0] / len(s) * 100, 2),
        f"cobertura_top{top_k}_%": round(vc.head(top_k).sum() / len(s) * 100, 2),
        "n_no_nulos": len(s),
    }

cat_cols = ["main_payment_type", "first_product_category",
            "customer_state", "first_seller_state", "order_status"]
pd.DataFrame([cat_concentration(mart, c) for c in cat_cols])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

top_cat = (delivered["first_product_category"].value_counts().head(15))
sns.barplot(x=top_cat.values, y=top_cat.index, ax=axes[0], hue=top_cat.index,
            palette="viridis", legend=False)
axes[0].set_title("Top 15 categorías de producto")
axes[0].set_xlabel("# de órdenes")

pay_counts = delivered["main_payment_type"].value_counts()
sns.barplot(x=pay_counts.values, y=pay_counts.index, ax=axes[1], hue=pay_counts.index,
            palette="rocket", legend=False)
axes[1].set_title("Método de pago principal")
axes[1].set_xlabel("# de órdenes")
for i, v in enumerate(pay_counts.values):
    axes[1].text(v, i, f"  {v/len(delivered)*100:.1f}%", va="center", fontsize=9)

top_states = delivered["customer_state"].value_counts().head(15)
sns.barplot(x=top_states.values, y=top_states.index, ax=axes[2], hue=top_states.index,
            palette="mako", legend=False)
axes[2].set_title("Top 15 estados del cliente (BR)")
axes[2].set_xlabel("# de órdenes")

plt.tight_layout(); plt.show()


### 6.4 Variable objetivo

Distribución de `review_score` y proporción de reseñas positivas.


In [ ]:
reviews = mart.dropna(subset=["review_score"]).copy()
reviews["review_score"] = reviews["review_score"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
score_counts = reviews["review_score"].value_counts().sort_index()
score_pct = (score_counts / score_counts.sum() * 100).round(2)
sns.barplot(x=score_counts.index, y=score_counts.values, ax=axes[0],
            hue=score_counts.index, palette="RdYlGn", legend=False)
axes[0].set_title("Distribución de review_score")
axes[0].set_xlabel("Puntaje"); axes[0].set_ylabel("# de reviews")
for i, (v, p) in enumerate(zip(score_counts.values, score_pct.values)):
    axes[0].text(i, v, f"{p}%", ha="center", va="bottom", fontsize=9)

pos_counts = reviews["is_positive_review"].value_counts()
labels = ["Negativa (≤3)", "Positiva (≥4)"]
axes[1].pie(pos_counts.reindex([0, 1]).values, labels=labels,
            autopct="%1.1f%%", colors=["#E57373", "#81C784"], startangle=90)
axes[1].set_title("¿La reseña es positiva?")
plt.tight_layout(); plt.show()

print(f"Total con review : {len(reviews):,}")
print(f"% positivas (≥4) : {reviews['is_positive_review'].mean()*100:.2f}%")
print(f"Asimetría review_score: {stats.skew(reviews['review_score']):.2f} "
      f"(negativa fuerte → mayoría 5 estrellas)")


## 7. Análisis temporal

¿Cómo evoluciona el volumen del marketplace? ¿Hay estacionalidad o
eventos atípicos?


In [ ]:
mart["purchase_date"] = mart["order_purchase_timestamp"].dt.date
daily = (mart.groupby("purchase_date").size().reset_index(name="n_orders"))
daily["purchase_date"] = pd.to_datetime(daily["purchase_date"])

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily["purchase_date"], daily["n_orders"], color="#1976D2", linewidth=1)
ax.set_title("Volumen diario de órdenes (Olist 2016-2018)")
ax.set_ylabel("# de órdenes"); ax.set_xlabel("Fecha"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Día con más órdenes: {daily.loc[daily['n_orders'].idxmax(), 'purchase_date'].date()} "
      f"({daily['n_orders'].max():,} órdenes)")


In [ ]:
mart["dow"]  = mart["order_purchase_timestamp"].dt.dayofweek
mart["hour"] = mart["order_purchase_timestamp"].dt.hour

dow_names = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
dow_counts = mart["dow"].value_counts().sort_index()
hour_counts = mart["hour"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(x=dow_names, y=dow_counts.values, ax=axes[0],
            hue=dow_names, palette="crest", legend=False)
axes[0].set_title("Órdenes por día de la semana"); axes[0].set_ylabel("# de órdenes")

sns.barplot(x=hour_counts.index, y=hour_counts.values, ax=axes[1],
            hue=hour_counts.index, palette="flare", legend=False)
axes[1].set_title("Órdenes por hora del día"); axes[1].set_xlabel("Hora")
plt.tight_layout(); plt.show()


## 8. Análisis geográfico

¿Cómo se distribuyen el ticket promedio y la satisfacción por estado?


In [ ]:
state_summary = (
    delivered.groupby("customer_state")
    .agg(n_orders=("order_id", "count"),
         avg_ticket=("total_value", "mean"),
         avg_delivery_days=("delivery_days_real", "mean"),
         pct_late=("is_late", "mean"),
         pct_positive=("is_positive_review", "mean"))
    .reset_index()
)
state_summary["pct_late"]     = (state_summary["pct_late"] * 100).round(2)
state_summary["pct_positive"] = (state_summary["pct_positive"] * 100).round(2)
state_summary["avg_ticket"]   = state_summary["avg_ticket"].round(2)
state_summary["avg_delivery_days"] = state_summary["avg_delivery_days"].round(2)
state_summary = state_summary.sort_values("n_orders", ascending=False)
state_summary.head(15)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
top10 = state_summary.head(10)
sns.barplot(data=top10, x="customer_state", y="avg_delivery_days", ax=axes[0],
            hue="customer_state", palette="rocket_r", legend=False)
axes[0].set_title("Tiempo medio de entrega por estado (top 10 por volumen)")
axes[0].set_ylabel("Días"); axes[0].set_xlabel("Estado")

worst_late = (state_summary[state_summary["n_orders"] > 100]
                .sort_values("pct_late", ascending=False).head(10))
sns.barplot(data=worst_late, x="customer_state", y="pct_late", ax=axes[1],
            hue="customer_state", palette="flare", legend=False)
axes[1].set_title("Estados con mayor tasa de retraso (vol > 100 órdenes)")
axes[1].set_ylabel("% órdenes con retraso"); axes[1].set_xlabel("Estado")
plt.tight_layout(); plt.show()


## 9. Análisis bivariante con el target

### 9.1 Numéricas vs. `is_positive_review`


In [ ]:
rev_df = delivered.dropna(subset=["is_positive_review"]).copy()
rev_df["is_positive_review"] = rev_df["is_positive_review"].astype(int)
rev_df["reseña"] = rev_df["is_positive_review"].map({0: "Negativa", 1: "Positiva"})

num_vs_target = ["delivery_days_real", "delivery_delay_days",
                 "total_value", "total_freight",
                 "n_items", "payment_installments_max"]
palette_target = {"Negativa": "#E57373", "Positiva": "#81C784"}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_vs_target):
    sub = rev_df[[col, "reseña"]].dropna()
    if sub[col].nunique() > 10:
        sub = sub[sub[col] <= sub[col].quantile(0.99)]
    sns.boxplot(data=sub, x="reseña", y=col, hue="reseña", ax=ax,
                palette=palette_target, legend=False, showfliers=False,
                order=["Negativa", "Positiva"])
    ax.set_title(col); ax.set_xlabel("")
plt.suptitle("Variables numéricas según el tipo de reseña",
             y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


### 9.2 Categóricas vs. `is_positive_review`

In [ ]:
def positive_rate_by(df, col, min_n=200, top=15):
    g = (df.groupby(col)
           .agg(n=("is_positive_review", "size"),
                pct_pos=("is_positive_review", "mean"))
           .reset_index())
    g = g[g["n"] >= min_n].copy()
    g["pct_pos"] = (g["pct_pos"] * 100).round(2)
    return (g.sort_values("pct_pos", ascending=False).head(top),
            g.sort_values("pct_pos", ascending=True).head(top))

best_cat, worst_cat = positive_rate_by(rev_df, "first_product_category")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=best_cat, x="pct_pos", y="first_product_category", ax=axes[0],
            hue="first_product_category", palette="Greens_r", legend=False)
axes[0].set_title("Top 15 categorías con MÁS reviews positivas")
axes[0].set_xlabel("% reviews positivas")
sns.barplot(data=worst_cat, x="pct_pos", y="first_product_category", ax=axes[1],
            hue="first_product_category", palette="Reds_r", legend=False)
axes[1].set_title("Top 15 categorías con MENOS reviews positivas")
axes[1].set_xlabel("% reviews positivas")
plt.tight_layout(); plt.show()


In [ ]:
pay_summary = (rev_df.groupby("main_payment_type")
                .agg(n=("is_positive_review", "size"),
                     pct_pos=("is_positive_review", "mean"))
                .reset_index())
pay_summary["pct_pos"] = (pay_summary["pct_pos"] * 100).round(2)
pay_summary = pay_summary.sort_values("pct_pos", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=pay_summary, x="main_payment_type", y="pct_pos", ax=ax,
            hue="main_payment_type", palette="viridis", legend=False)
ax.set_title("% de reviews positivas por método de pago principal")
ax.set_ylabel("% positivas"); ax.set_xlabel("Método de pago")
ax.axhline(rev_df["is_positive_review"].mean() * 100, color="red",
           linestyle="--", linewidth=1, label="Promedio global")
for i, (n, p) in enumerate(zip(pay_summary["n"], pay_summary["pct_pos"])):
    ax.text(i, p + 0.5, f"n={n:,}", ha="center", fontsize=8)
ax.legend()
plt.tight_layout(); plt.show()


### 9.3 Mapa de correlaciones (Spearman)

In [ ]:
corr_cols = ["n_items", "n_distinct_products", "n_distinct_sellers",
             "total_price", "total_freight", "total_value", "avg_item_price",
             "n_payments", "payment_total", "payment_installments_max",
             "delivery_days_real", "delivery_delay_days",
             "is_late", "is_positive_review"]
corr = rev_df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.6}, ax=ax)
ax.set_title("Correlaciones de Spearman entre variables numéricas",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## 10. Análisis multivariante

Las correlaciones por pares ayudan, pero algunas relaciones solo se
ven combinando **tres o más variables**. En esta sección hacemos:

- 10.1 Análisis condicional (3 variables): cómo cambia el target
  cuando cruzamos dos features.
- 10.2 PCA (Análisis de Componentes Principales) para visualizar la
  estructura conjunta de las variables numéricas.
- 10.3 Heatmap cruzado *categoría × estado* del % positivo.
- 10.4 Correlaciones intra-grupo (¿la misma relación numérica se
  comporta igual en órdenes positivas y negativas?).


### 10.1 Análisis condicional

¿Cómo varía la **probabilidad de reseña positiva** según
combinaciones de **tiempo de entrega** y **número de cuotas**?


In [ ]:
sub = rev_df.dropna(subset=["delivery_days_real", "payment_installments_max"]).copy()

# Bins de tiempo de entrega y de cuotas
sub["bucket_delivery"] = pd.cut(
    sub["delivery_days_real"],
    bins=[-np.inf, 5, 10, 15, 20, 30, np.inf],
    labels=["≤5d", "6-10d", "11-15d", "16-20d", "21-30d", ">30d"],
)
sub["bucket_install"] = pd.cut(
    sub["payment_installments_max"],
    bins=[0, 1, 3, 6, 10, np.inf],
    labels=["1", "2-3", "4-6", "7-10", "11+"],
)

heat = (sub.groupby(["bucket_delivery", "bucket_install"], observed=True)
           ["is_positive_review"].mean().unstack() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn", center=77,
            cbar_kws={"label": "% reviews positivas"}, ax=ax)
ax.set_title("% reseñas positivas según tiempo de entrega × # de cuotas")
ax.set_xlabel("# de cuotas"); ax.set_ylabel("Tiempo de entrega")
plt.tight_layout(); plt.show()


**Lectura.** El gradiente vertical es muy claro: a más días de
entrega, menos satisfacción. El efecto del número de cuotas es más
sutil pero también monótono — a más cuotas, ligeramente menos
satisfacción para la misma franja de entrega.


### 10.2 PCA — estructura conjunta de las variables numéricas

Reducimos 8 features a 2 componentes principales y coloreamos por el
target. Si los grupos se separan, hay señal para clasificación.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca_features = ["delivery_days_real", "delivery_delay_days",
                "total_value", "total_freight",
                "n_items", "n_distinct_sellers",
                "payment_total", "payment_installments_max"]
pca_df = rev_df[pca_features + ["is_positive_review"]].dropna()

# Para que el scatter sea legible muestreamos 8000 puntos
sample = pca_df.sample(min(8000, len(pca_df)), random_state=42)

X = StandardScaler().fit_transform(sample[pca_features])
pca = PCA(n_components=2, random_state=42).fit(X)
Z = pca.transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = sample["is_positive_review"].map({0: "Negativa", 1: "Positiva"})
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=labels, ax=axes[0],
                palette={"Negativa": "#E57373", "Positiva": "#81C784"},
                alpha=0.4, s=10)
axes[0].set_title(f"PCA — proyección 2D (varianza explicada: "
                  f"{pca.explained_variance_ratio_.sum()*100:.1f}%)")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")

# Loadings (cómo contribuye cada feature a las componentes)
loadings = pd.DataFrame(pca.components_.T, columns=["PC1", "PC2"], index=pca_features)
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("Loadings — peso de cada variable en cada componente")
plt.tight_layout(); plt.show()

print(f"Varianza explicada por PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"Varianza explicada por PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")


**Lectura del PCA**

- **PC1** captura principalmente las variables monetarias
  (`total_value`, `total_freight`, `payment_total`): es un eje de
  *"tamaño del ticket"*.
- **PC2** captura las variables logísticas (`delivery_days_real`,
  `delivery_delay_days`): es un eje de *"experiencia de entrega"*.
- La separación de los dos grupos del target no es perfecta — se
  solapan mucho — lo que confirma que **un solo eje no basta**: el
  modelo necesitará usar varias variables en simultáneo.


### 10.3 Heatmap cruzado — Categoría × Estado

¿Hay categorías que rinden mucho mejor o peor según el estado del
cliente? (riesgo de fricciones logísticas regionales)


In [ ]:
top_cats   = rev_df["first_product_category"].value_counts().head(8).index
top_states_v = rev_df["customer_state"].value_counts().head(8).index
sub_cs = rev_df[
    rev_df["first_product_category"].isin(top_cats) &
    rev_df["customer_state"].isin(top_states_v)
]
mat = (sub_cs.groupby(["first_product_category", "customer_state"], observed=True)
              ["is_positive_review"].mean().unstack() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(mat, annot=True, fmt=".1f", cmap="RdYlGn", center=77,
            cbar_kws={"label": "% positivas"}, ax=ax)
ax.set_title("% reseñas positivas — Top 8 categorías × Top 8 estados")
plt.tight_layout(); plt.show()


### 10.4 Correlaciones intra-grupo

¿La relación entre las variables numéricas es la **misma** dentro de
las órdenes positivas y dentro de las negativas, o cambia? Si cambia,
significa que hay **interacciones** que un modelo no lineal podría
capturar mejor que un lineal.


In [ ]:
pairs = [
    ("delivery_days_real", "is_positive_review"),
    ("total_value", "is_positive_review"),
    ("payment_installments_max", "is_positive_review"),
    ("total_freight", "delivery_days_real"),
]

intragroup = []
for a, b in pairs:
    overall = rev_df[[a, b]].dropna().corr(method="spearman").iloc[0, 1]
    pos = rev_df[rev_df["is_positive_review"] == 1][[a, b]].dropna().corr(method="spearman").iloc[0, 1]
    neg = rev_df[rev_df["is_positive_review"] == 0][[a, b]].dropna().corr(method="spearman").iloc[0, 1]
    intragroup.append({"par": f"{a} ↔ {b}",
                        "global":   round(overall, 3),
                        "positiva": round(pos, 3),
                        "negativa": round(neg, 3),
                        "delta":    round(pos - neg, 3)})

pd.DataFrame(intragroup)


**Lectura.** Diferencias importantes entre `positiva` y `negativa`
revelan **interacciones no lineales** entre features. Por ejemplo, si
`total_freight ↔ delivery_days_real` es más fuerte en órdenes
negativas, significa que el costo de envío explica mejor los retrasos
en las experiencias malas.


## 11. Validación estadística de las hipótesis

Aplicamos los tests definidos en la sección 3 con α = 0.05.


In [ ]:
alpha = 0.05
resultados = []

# H1: Mann-Whitney U para delivery_days_real entre positivas y negativas
g1 = rev_df.loc[rev_df["is_positive_review"] == 1, "delivery_days_real"].dropna()
g0 = rev_df.loc[rev_df["is_positive_review"] == 0, "delivery_days_real"].dropna()
u_stat, p_h1 = stats.mannwhitneyu(g1, g0, alternative="two-sided")
resultados.append({
    "Hipótesis": "H1: delivery_days_real difiere por reseña",
    "Test": "Mann-Whitney U",
    "Estadístico": f"U = {u_stat:,.0f}",
    "p-valor": f"{p_h1:.2e}",
    "Decisión": "Rechazar H0" if p_h1 < alpha else "No rechazar H0",
})

# H2: Chi-cuadrado is_late vs is_positive_review
ct_h2 = pd.crosstab(rev_df["is_late"], rev_df["is_positive_review"])
chi2, p_h2, dof, exp = stats.chi2_contingency(ct_h2)
resultados.append({
    "Hipótesis": "H2: is_late vs is_positive_review",
    "Test": "Chi-cuadrado independencia",
    "Estadístico": f"χ² = {chi2:,.2f} (gl={dof})",
    "p-valor": f"{p_h2:.2e}",
    "Decisión": "Rechazar H0" if p_h2 < alpha else "No rechazar H0",
})

# H3: Chi-cuadrado main_payment_type vs is_positive_review
ct_h3 = pd.crosstab(rev_df["main_payment_type"], rev_df["is_positive_review"])
chi2, p_h3, dof, exp = stats.chi2_contingency(ct_h3)
resultados.append({
    "Hipótesis": "H3: main_payment_type vs is_positive_review",
    "Test": "Chi-cuadrado independencia",
    "Estadístico": f"χ² = {chi2:,.2f} (gl={dof})",
    "p-valor": f"{p_h3:.2e}",
    "Decisión": "Rechazar H0" if p_h3 < alpha else "No rechazar H0",
})

# H4: Spearman entre cuotas y review_score
sub_h4 = rev_df[["payment_installments_max", "review_score"]].dropna()
rho, p_h4 = stats.spearmanr(sub_h4["payment_installments_max"], sub_h4["review_score"])
resultados.append({
    "Hipótesis": "H4: cuotas ↔ review_score",
    "Test": "Spearman",
    "Estadístico": f"ρ = {rho:.3f}",
    "p-valor": f"{p_h4:.2e}",
    "Decisión": "Rechazar H0" if p_h4 < alpha else "No rechazar H0",
})

# H5: Mann-Whitney delivery_days_real sudeste vs resto
sudeste = {"SP", "RJ", "MG", "ES"}
sub_h5 = rev_df[["delivery_days_real", "customer_state"]].dropna()
g_se = sub_h5.loc[sub_h5["customer_state"].isin(sudeste), "delivery_days_real"]
g_rs = sub_h5.loc[~sub_h5["customer_state"].isin(sudeste), "delivery_days_real"]
u_stat, p_h5 = stats.mannwhitneyu(g_se, g_rs, alternative="two-sided")
resultados.append({
    "Hipótesis": "H5: delivery_days_real sudeste vs resto",
    "Test": "Mann-Whitney U",
    "Estadístico": f"U = {u_stat:,.0f}",
    "p-valor": f"{p_h5:.2e}",
    "Decisión": "Rechazar H0" if p_h5 < alpha else "No rechazar H0",
})

resultados_df = pd.DataFrame(resultados)
resultados_df


**Lectura de los tests**

- Las hipótesis con `p < 0.05` confirman estadísticamente lo que el
  análisis visual sugería: hay una asociación real entre logística y
  satisfacción, así como diferencias geográficas significativas.
- El tamaño del efecto se debe interpretar **junto con el p-valor**:
  con casi 100k observaciones es fácil obtener significancia
  estadística incluso con efectos pequeños. Por eso reportamos también
  el estadístico (Spearman ρ, Mann-Whitney U) que mide la magnitud.


## 12. Detección de outliers y plan de tratamiento

### 12.1 Métodos de detección

Aplicamos **tres reglas complementarias** sobre cada variable
numérica relevante:

- **IQR (1.5×)** — clásico de Tukey: outlier si está fuera de
  `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.
- **IQR (3.0×)** — versión estricta para "outliers extremos".
- **Z-score modificado (MAD)** — robusto a colas largas; outlier si
  `|0.6745·(x − mediana)/MAD| > 3.5`.


In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def mad_outliers(s, threshold=3.5):
    med = s.median()
    mad = np.median(np.abs(s - med))
    if mad == 0:
        return pd.Series([False] * len(s), index=s.index)
    z_mod = 0.6745 * (s - med) / mad
    return z_mod.abs() > threshold

vars_outlier = ["delivery_days_real", "delivery_delay_days",
                "total_value", "total_freight", "payment_total",
                "payment_installments_max", "n_items"]

rows = []
for col in vars_outlier:
    s = delivered[col].dropna()
    lb15, ub15 = iqr_bounds(s, 1.5)
    lb30, ub30 = iqr_bounds(s, 3.0)
    n_iqr15 = ((s < lb15) | (s > ub15)).sum()
    n_iqr30 = ((s < lb30) | (s > ub30)).sum()
    n_mad   = mad_outliers(s).sum()
    rows.append({
        "variable": col,
        "n": len(s),
        "min": round(s.min(), 2), "max": round(s.max(), 2),
        "IQR_1.5_lim_inf": round(lb15, 2), "IQR_1.5_lim_sup": round(ub15, 2),
        "n_outliers_IQR_1.5": int(n_iqr15),
        "%_IQR_1.5": round(n_iqr15 / len(s) * 100, 2),
        "n_outliers_IQR_3.0": int(n_iqr30),
        "%_IQR_3.0": round(n_iqr30 / len(s) * 100, 2),
        "n_outliers_MAD":  int(n_mad),
        "%_MAD":  round(n_mad / len(s) * 100, 2),
    })
out_df = pd.DataFrame(rows)
out_df


In [ ]:
# Boxplots con bigotes extendidos para visualizar
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, vars_outlier):
    sns.boxplot(x=delivered[col].dropna(), ax=ax, color="#1976D2", fliersize=2)
    ax.set_title(col); ax.set_xlabel("")
# Esconder el último subplot vacío
if len(vars_outlier) < axes.size:
    for k in range(len(vars_outlier), axes.size):
        axes.flat[k].axis("off")
plt.suptitle("Boxplots — visualización de outliers",
             y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


### 12.2 Reglas de negocio adicionales

Más allá de lo estadístico, hay valores que son **inválidos por
diseño** y deben tratarse aparte:

- `delivery_days_real < 0` → entrega antes de la compra (error de
  captura, no es un outlier "real").
- `delivery_days_real > 60` → entrega muy tardía: outlier estadístico
  pero **válido como evento de negocio** (señal fuerte de mal
  servicio).
- `total_value > p99.9` → tickets gigantes: probables compras B2B o
  productos de muy alto valor. No son errores, pero sí pueden dominar
  modelos lineales.
- `payment_installments_max > 12` → cuotas extremas: clientes con
  capacidad de financiación amplia. Bajo volumen, conviene
  considerarlas categóricamente.


### 12.3 Plan concreto de tratamiento de outliers

Cada acción es **trazable** y se ejecutará al inicio del notebook de
modelado (Fase 4). No se modifican los CSV procesados — el plan se
aplica únicamente sobre el conjunto de entrenamiento.

| Variable | Regla detectada | Umbral | Acción | Justificación |
|---|---|---|---|---|
| `delivery_days_real` | inconsistencia | < 0 | **Eliminar** la fila | Error de captura, no representa nada real. |
| `delivery_days_real` | cola larga válida | > 60 días | **Conservar** sin transformar | Señal genuina de mala experiencia, esencial para el modelo. |
| `delivery_delay_days` | cola larga válida | > 30 días | **Conservar** sin transformar | Idem, señal predictiva. |
| `total_value` | cola monetaria | > p99 (~R$1.000) | **Winsorizar** al p99 | Evita que tickets gigantes dominen modelos lineales. |
| `total_freight` | cola monetaria | > p99 | **Winsorizar** al p99 | Mismo criterio. |
| `payment_total` | cola monetaria | > p99 | **Winsorizar** al p99 | Mismo criterio. |
| `payment_installments_max` | discreta extrema | > 12 | **Re-categorizar** a `12+` | Conserva la información sin valores numéricos extremos. |
| `n_items` | discreta extrema | > 10 | **Re-categorizar** a `10+` | Idem. |
| `is_late` | flag binaria | — | **Conservar** | Es un objetivo predictivo intermedio. |

**Pseudocódigo del plan de limpieza** (a aplicar en Fase 4):

```python
df = df[df["delivery_days_real"] >= 0]                      # eliminar inconsistencias
for c in ["total_value", "total_freight", "payment_total"]:
    p99 = df[c].quantile(0.99)
    df[c] = df[c].clip(upper=p99)                           # winsorización
df["payment_installments_max"] = df["payment_installments_max"].clip(upper=12)
df["n_items"] = df["n_items"].clip(upper=10)
```


## 13. Insights de negocio (top hallazgos)

| # | Insight | Evidencia | Implicación de negocio |
|---|---|---|---|
| 1 | El **77 %** de las reseñas son positivas (≥4 estrellas). | Sección 6.4 | Target desbalanceado; se requieren métricas adecuadas (F1, ROC-AUC) y técnicas como class weighting. |
| 2 | La **logística es el driver dominante** de la satisfacción, **confirmado estadísticamente** (H1, H2). | Secciones 9.1, 11 | Optimizar plazos de entrega es la palanca de mayor retorno sobre satisfacción. |
| 3 | El **sudeste tiene mejor logística** que el resto del país (H5 rechazada). | Secciones 8, 11 | Oportunidad de expansión logística regional. |
| 4 | El **método de pago se asocia** con la satisfacción (H3 rechazada). | Secciones 9.2, 11 | Considerar estrategias diferenciadas por método de pago. |
| 5 | Existe una **correlación negativa débil pero significativa** entre cuotas y review_score (H4). | Sección 11 | Clientes que financian más son más exigentes; vale la pena segmentar la experiencia. |
| 6 | **Black Friday 2017** generó un pico extraordinario; el modelo debe ser robusto a estacionalidades. | Sección 7 | Considerar variables de calendario en el modelo. |
| 7 | El PCA muestra que **un eje no basta**: el modelo debe combinar logística y monto. | Sección 10.2 | Preferir modelos no lineales (Random Forest, XGBoost) sobre regresión lineal. |
| 8 | **Plan de outliers documentado**: eliminación de inconsistencias + winsorización al p99 + re-categorización de valores discretos extremos. | Sección 12.3 | Reduce ruido sin perder señal predictiva. |
| 9 | `total_value`, `total_price` y `payment_total` son **redundantes** (ρ > 0.95). | Sección 9.3 | Mantener una sola variable monetaria en el modelo. |


## 14. Conclusiones y siguiente fase

**Conclusiones**

- Los KPIs muestran un marketplace en crecimiento con CSAT del ~77 % y
  oportunidades claras en logística (OTIF < 95 %) y diversificación
  geográfica (concentración top-3 > 60 %).
- Las cinco hipótesis fueron contrastadas estadísticamente; los
  resultados confirman la intuición de negocio y dan respaldo formal
  a las features candidatas.
- El plan de tratamiento de outliers está **explícito y reproducible**,
  listo para aplicarse en la Fase 4.

**Siguiente fase (Fase 3 — BI)**

- Construir un dashboard que materialice los KPIs anteriores con
  filtros por estado, mes y categoría.

**Siguiente fase (Fase 4 — Modelado predictivo)**

- Aplicar el plan de outliers de la sección 12.3.
- Variables candidatas iniciales:
  *logísticas* (`delivery_days_real`, `delivery_delay_days`, `is_late`),
  *económicas* (`total_value` o `payment_total`, `total_freight`,
  `payment_installments_max`),
  *de orden* (`n_items`, `n_distinct_sellers`),
  *categóricas* (`first_product_category`, `customer_state`,
  `main_payment_type`).
- Probar modelos no lineales (Random Forest, XGBoost) por la
  evidencia del PCA y del análisis intra-grupo.
- Métricas: F1 y ROC-AUC sobre `is_positive_review`, comparando
  contra el baseline trivial (~77 %).
